# TabFM vs Meridian MMM (synthetic PoC)

> **Disclaimer:** All values below are **synthetic / dummy**. They are **not**
> estimates of real campaign performance and must not be used for budgeting or
> other business decisions.

This notebook runs the same comparison as `scripts/run_comparison.py`:

1. Load Meridian-shaped weekly dummy CSV (`data/hk_skincare_mmm_dummy.csv`)
2. Time-split: earlier weeks = train/context, later weeks = holdout
3. Score **TabFM** (or dry-run mock) and **Meridian / Meridian-style** on holdout KPI
4. Optionally report channel contribution recovery vs planted `contribution_*`

**TabFM weight license:** default Hugging Face weights are
`tabfm-non-commercial-v1.0` (non-commercial / non-production).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "mmm_compare").exists():
    # Allow running with notebook cwd inside a subfolder
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from mmm_compare.compare import print_table, run_comparison
from mmm_compare.data import load_mmm_dataset

DATA = ROOT / "data" / "hk_skincare_mmm_dummy.csv"
RESULTS = ROOT / "results"

# Set True to skip TabFM weight download / Meridian MCMC
DRY_RUN = True

print("Repo root:", ROOT)
print("Data:", DATA, "exists=", DATA.exists())

In [ ]:
data = load_mmm_dataset(DATA)
print("KPI:", data.kpi_col)
print("Channels:", data.channel_keys)
print("Train weeks:", data.train_idx, "| Holdout weeks:", data.test_idx)
print("Features:", data.feature_cols)
display(data.frame.head())

In [ ]:
table, results, payload = run_comparison(
    DATA,
    dry_run=DRY_RUN,
    prefer_real_meridian=True,
    results_dir=RESULTS,
)
print_table(table)
display(table)
print("Disclaimer:", payload["disclaimer"])
print("TabFM mode:", results["tabfm"].mode, results["tabfm"].extras)
print("Meridian mode:", results["meridian"].mode, results["meridian"].extras.get("note", ""))

In [ ]:
# Holdout predictions vs actual (synthetic KPI)
import matplotlib.pyplot as plt

y = data.y_test.to_numpy()
weeks = data.frame.loc[data.test_idx, "time"]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(weeks, y, marker="o", label="actual (synthetic)")
ax.plot(weeks, results["tabfm"].y_pred_test, marker="s", label=f"TabFM ({results['tabfm'].mode})")
ax.plot(
    weeks,
    results["meridian"].y_pred_test,
    marker="^",
    label=f"Meridian ({results['meridian'].mode})",
)
ax.set_title("Holdout KPI predictions — synthetic data only")
ax.set_ylabel(data.kpi_col)
ax.tick_params(axis="x", rotation=45)
ax.legend()
fig.tight_layout()
plt.show()

## How to run with real TabFM weights

1. `bash scripts/install_deps.sh`
2. Optionally export `HF_TOKEN` if Hub auth is required
3. Set `DRY_RUN = False` above and re-run, or:
   `python scripts/run_comparison.py -v`

For full Meridian: `INSTALL_MERIDIAN=1 bash scripts/install_deps.sh`